# RoofInsight: train RF-DETR on Google Colab

Trains the roof damage detector (classes `0 = wind`, `1 = hail`, `2 = missing_shingles`) on `roof_damage_rfdetr.zip`.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** (or better).
2. In Google Drive, make a folder `RoofInsight` and put in it:
   * `roof_damage_rfdetr.zip` (the dataset)
   * `training/train.py` (from this repository / the migration bundle)
3. Run the cells top to bottom.

Checkpoints go to `MyDrive/RoofInsight/runs/<RUN_NAME>/`. `last.ckpt` is rewritten after every epoch.
**If Colab disconnects, reconnect and run every cell again with the same `RUN_NAME`: training resumes where it stopped.**

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings
Change these, then run the rest. Keep `RUN_NAME` the same to resume; change it to start a new run.

In [ ]:
DRIVE_DIR   = '/content/drive/MyDrive/RoofInsight'
DATASET_ZIP = f'{DRIVE_DIR}/roof_damage_rfdetr.zip'
TRAIN_PY    = f'{DRIVE_DIR}/training/train.py'

RUN_NAME    = 'medium-run-001'
MODEL       = 'medium'   # nano | small | medium | base | large  (nano = same size as v1)
EPOCHS      = 50
BATCH_SIZE  = 4          # lower to 2 (and GRAD_ACCUM to 8) if you get CUDA out of memory
GRAD_ACCUM  = 4          # effective batch = BATCH_SIZE x GRAD_ACCUM = 16
CHECKPOINT_INTERVAL = 10 # keep an archive .ckpt every N epochs (~0.5 GB each); last.ckpt is saved every epoch
PRETRAIN    = ''         # optional: path to a .pth to fine-tune from (same MODEL size), e.g. the v1 nano weights

CHECKPOINT_DIR = f'{DRIVE_DIR}/runs/{RUN_NAME}'
LOCAL_DATA     = '/content/data'   # dataset is unzipped to fast local disk, not Drive
import os
for path in (DATASET_ZIP, TRAIN_PY):
    print(('OK      ' if os.path.exists(path) else 'MISSING ') + path)

In [ ]:
!pip install -q "rfdetr[train]==1.11.2"
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))

## Check the dataset
Unzips once to `/content/data` and validates classes, files and boxes. No training yet.

In [ ]:
!python "{TRAIN_PY}" --dataset "{DATASET_ZIP}" --extract-to "{LOCAL_DATA}" --check-only

## Train (resumes automatically)
A T4 takes very roughly 1 to 3 minutes per epoch for `medium` on this dataset; measure the first epoch and plan from it.

In [ ]:
pretrain = f'--pretrain "{PRETRAIN}"' if PRETRAIN else ''
!python "{TRAIN_PY}" --dataset "{DATASET_ZIP}" --extract-to "{LOCAL_DATA}" --output "{CHECKPOINT_DIR}" \
    --model {MODEL} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --grad-accum-steps {GRAD_ACCUM} \
    --checkpoint-interval {CHECKPOINT_INTERVAL} {pretrain}

## Results
`checkpoint_best_total.pth` is the weights file to keep. Test-set metrics are printed at the end of training and saved in `metrics.csv`.

In [ ]:
import pandas as pd
m = pd.read_csv(f'{CHECKPOINT_DIR}/metrics.csv')
cols = [c for c in ['epoch', 'val/ema_mAP_50', 'val/ema_mAP_50_95', 'val/F1', 'test/mAP_50', 'test/mAP_50_95', 'test/F1'] if c in m]
m[cols].groupby('epoch').max().tail(10)

## Try the trained model on one test photo

In [ ]:
import glob, rfdetr, supervision as sv
from PIL import Image
MODEL_CLASS = {'nano': rfdetr.RFDETRNano, 'small': rfdetr.RFDETRSmall, 'medium': rfdetr.RFDETRMedium,
               'base': rfdetr.RFDETRBase, 'large': rfdetr.RFDETRLarge}[MODEL]
model = MODEL_CLASS(pretrain_weights=f'{CHECKPOINT_DIR}/checkpoint_best_total.pth')
names = {0: 'wind', 1: 'hail', 2: 'missing_shingles'}
photo = sorted(glob.glob(f'{LOCAL_DATA}/*/test/*.jpg'))[0]
image = Image.open(photo)
det = model.predict(image, threshold=0.4)
labels = [f'{names[c]} {s:.2f}' for c, s in zip(det.class_id, det.confidence)]
out = sv.BoxAnnotator().annotate(image.copy(), det)
out = sv.LabelAnnotator().annotate(out, det, labels)
display(out)